# Scene Shot Detection with PySceneDetect

Detect scene cuts (shots) in a video and build a list of every scene with start/end timecodes, frame numbers, and duration.

Optionally:
- Save a JPEG/PNG thumbnail for each scene
- Transcribe the audio **inside each scene's exact time window** with Whisper, so transcript and image use the same shot boundaries

**Requirements:** `scenedetect`, OpenCV (`opencv-python`), `openai-whisper`, and FFmpeg.

In [ ]:
# Pin NumPy 1.x — Whisper + PyTorch break on NumPy 2.x in this environment.
# After running this cell once: Kernel -> Restart Kernel, then Run All.
%pip install -q "numpy==1.26.4" "scenedetect[opencv]" openai-whisper

In [ ]:
from pathlib import Path

import pandas as pd
from scenedetect import detect, open_video, ContentDetector, AdaptiveDetector, save_images

# --- Configuration ---
VIDEO_PATH = "sample_video.mp4"  # change this
OUTPUT_CSV = None  # e.g. "scene_list.csv" to save results

# Image export (one thumbnail per scene by default)
SAVE_IMAGES = True
OUTPUT_IMAGE_DIR = "scene_shots"  # folder created next to the notebook
NUM_IMAGES_PER_SCENE = 1  # 1 = one frame per shot; 3 = start/middle/end
IMAGE_EXTENSION = "jpg"  # jpg, png, or webp

# Whisper transcription (audio sliced per scene, same timestamps as images)
TRANSCRIBE_SCENES = True
WHISPER_MODEL = "medium"  # tiny, base, small, medium, large
WHISPER_LANGUAGE = "en"  # e.g. "en", "ar", or None for auto-detect
MIN_SCENE_AUDIO_SECONDS = 0.3  # skip transcription for very short/silent scenes
OUTPUT_TRANSCRIPT_JSON = "scene_transcripts.json"  # set None to skip saving

# Detector options:
# - ContentDetector(): fast cuts (default, good for most videos)
# - AdaptiveDetector(): often better for fast motion or noisy footage
DETECTOR = ContentDetector(threshold=27.0)

SHOW_PROGRESS = True
START_TIME = None  # e.g. "00:01:00" or 60.0 (seconds) or 1500 (frames)
END_TIME = None    # e.g. "00:05:00"

In [ ]:
video_path = Path(VIDEO_PATH).expanduser().resolve()
if not video_path.is_file():
    raise FileNotFoundError(f"Video not found: {video_path}")

print(f"Analyzing: {video_path}")
print(f"Detector: {DETECTOR.__class__.__name__}")

In [ ]:
scenes = detect(
    str(video_path),
    DETECTOR,
    show_progress=SHOW_PROGRESS,
    start_time=START_TIME,
    end_time=END_TIME,
    start_in_scene=True,
)

print(f"Found {len(scenes)} scene(s)")

In [ ]:
def scenes_to_records(scene_list):
    """Convert PySceneDetect scene tuples into plain dict records."""
    records = []
    for i, (start, end) in enumerate(scene_list, start=1):
        duration_seconds = end.get_seconds() - start.get_seconds()
        records.append(
            {
                "scene": i,
                "start_timecode": start.get_timecode(),
                "end_timecode": end.get_timecode(),
                "start_seconds": round(start.get_seconds(), 3),
                "end_seconds": round(end.get_seconds(), 3),
                "duration_seconds": round(duration_seconds, 3),
                "start_frame": start.get_frames(),
                "end_frame": end.get_frames(),
                "frame_count": end.get_frames() - start.get_frames(),
            }
        )
    return records


def attach_scene_images(scene_records, image_paths_by_scene, image_dir):
    """Map save_images output onto 1-based scene numbers.

    PySceneDetect returns 0-based dict keys, while filenames use Scene-001, Scene-002, ...
    """
    for record in scene_records:
        paths = image_paths_by_scene.get(record["scene"] - 1, [])
        record["image_paths"] = [str(image_dir / Path(path).name) for path in paths]
        record["image_path"] = record["image_paths"][0] if record["image_paths"] else None


scene_records = scenes_to_records(scenes)
scene_df = pd.DataFrame(scene_records)
scene_df

In [ ]:
image_paths_by_scene = {}

if SAVE_IMAGES:
    image_dir = Path(OUTPUT_IMAGE_DIR).expanduser().resolve()
    image_dir.mkdir(parents=True, exist_ok=True)

    video = open_video(str(video_path))
    image_paths_by_scene = save_images(
        scene_list=scenes,
        video=video,
        num_images=NUM_IMAGES_PER_SCENE,
        output_dir=str(image_dir),
        image_extension=IMAGE_EXTENSION,
        show_progress=SHOW_PROGRESS,
    )

    attach_scene_images(scene_records, image_paths_by_scene, image_dir)
    scene_df = pd.DataFrame(scene_records)
    print(f"Saved images to {image_dir}")
    print(f"Scene 1 image: {scene_records[0].get('image_path')}")
else:
    print("Image export disabled. Set SAVE_IMAGES = True to save shot thumbnails.")

In [ ]:
# Preview scene metadata (re-run the final cell after Whisper for transcripts)
scene_list = scene_records
scene_list[:3]

## Whisper transcription per scene

Audio is sliced with the **exact same** `start_seconds` / `end_seconds` used for scene detection and shot images. Whisper transcribes only what is heard during that shot.

In [ ]:
import json
from typing import Optional, Tuple

import numpy as np

# Whisper fails if NumPy 2.x is already loaded in this kernel session.
if int(np.__version__.split(".")[0]) >= 2:
    raise RuntimeError(
        f"NumPy {np.__version__} is incompatible with Whisper here. "
        "Run the install cell, then use Kernel -> Restart Kernel, and Run All."
    )

import whisper
from whisper.audio import SAMPLE_RATE


def slice_scene_audio(audio: np.ndarray, start_seconds: float, end_seconds: float) -> np.ndarray:
    start_sample = int(round(start_seconds * SAMPLE_RATE))
    end_sample = int(round(end_seconds * SAMPLE_RATE))
    return audio[start_sample:end_sample]


def transcribe_scene_clip(model, audio: np.ndarray, scene_record: dict) -> Tuple[str, Optional[str]]:
    clip = slice_scene_audio(
        audio,
        scene_record["start_seconds"],
        scene_record["end_seconds"],
    )

    duration = len(clip) / SAMPLE_RATE
    if duration < MIN_SCENE_AUDIO_SECONDS:
        return "", None

    transcribe_kwargs = {
        "condition_on_previous_text": False,
        "verbose": False,
    }
    if WHISPER_LANGUAGE:
        transcribe_kwargs["language"] = WHISPER_LANGUAGE

    result = model.transcribe(clip, **transcribe_kwargs)
    return result.get("text", "").strip(), result.get("language")

In [ ]:
if TRANSCRIBE_SCENES:
    print(f"Loading Whisper model: {WHISPER_MODEL}")
    whisper_model = whisper.load_model(WHISPER_MODEL)

    print(f"Loading audio from: {video_path.name}")
    full_audio = whisper.load_audio(str(video_path))

    detected_language = WHISPER_LANGUAGE
    for record in scene_records:
        transcript, language = transcribe_scene_clip(whisper_model, full_audio, record)
        record["transcript"] = transcript
        if detected_language is None and language:
            detected_language = language

    for record in scene_records:
        record["transcript_language"] = detected_language

    scene_df = pd.DataFrame(scene_records)
    print(f"Transcribed {len(scene_records)} scene(s)")
else:
    print("Whisper transcription disabled. Set TRANSCRIBE_SCENES = True to enable.")

In [ ]:
# Preview transcripts aligned to scene timecodes
if TRANSCRIBE_SCENES:
    for record in scene_records:
        print(
            f"Scene {record['scene']} "
            f"[{record['start_timecode']} -> {record['end_timecode']}]: "
            f"{record.get('transcript', '') or '(no speech)'}"
        )

In [ ]:
# Scene boundary consistency check
if TRANSCRIBE_SCENES:
    tolerance = 0.05
    gaps = []

    for prev, curr in zip(scene_records, scene_records[1:]):
        gap = round(curr["start_seconds"] - prev["end_seconds"], 3)
        if abs(gap) > tolerance:
            gaps.append((prev["scene"], curr["scene"], gap))

    if gaps:
        for prev_scene, curr_scene, gap in gaps:
            print(f"Warning: gap between scene {prev_scene} and {curr_scene}: {gap}s")
    else:
        print("Scene boundaries are contiguous.")

    spoken_scenes = sum(1 for record in scene_records if record.get("transcript"))
    print(f"Scenes with speech: {spoken_scenes}/{len(scene_records)}")

In [ ]:
if TRANSCRIBE_SCENES and OUTPUT_TRANSCRIPT_JSON:
    output_path = Path(OUTPUT_TRANSCRIPT_JSON).expanduser().resolve()
    payload = {
        "video_path": str(video_path),
        "language": detected_language,
        "scenes": [
            {
                "scene": record["scene"],
                "start_timecode": record["start_timecode"],
                "end_timecode": record["end_timecode"],
                "start_seconds": record["start_seconds"],
                "end_seconds": record["end_seconds"],
                "transcript": record.get("transcript", ""),
                "transcript_source": record.get("transcript_source", ""),
                "image_path": record.get("image_path"),
            }
            for record in scene_records
        ],
    }
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    print(f"Saved transcripts to {output_path}")

In [ ]:
# Final scene table (video cuts + transcript on the same timeline)
if TRANSCRIBE_SCENES:
    display_columns = [
        "scene",
        "start_timecode",
        "end_timecode",
        "duration_seconds",
        "transcript",
        "transcript_source",
        "image_path",
    ]
    scene_df = pd.DataFrame(scene_records)
    scene_list = scene_records
    scene_df[display_columns]

In [ ]:
if OUTPUT_CSV:
    from scenedetect import write_scene_list

    output_path = Path(OUTPUT_CSV).expanduser().resolve()
    write_scene_list(str(output_path), scenes, video_path=str(video_path))
    print(f"Saved scene list to {output_path}")
else:
    # Save with pandas if you prefer a simple CSV without PySceneDetect helpers
    # scene_df.to_csv("scene_list.csv", index=False)
    print("Set OUTPUT_CSV to save results (or uncomment scene_df.to_csv above).")

## Optional: try AdaptiveDetector

If `ContentDetector` misses cuts or produces too many false positives, rerun detection with `AdaptiveDetector()` instead.

In [ ]:
# Preview image + transcript together for each scene
if SAVE_IMAGES:
    from IPython.display import display, Image

    for record in scene_records:
        source = record.get("transcript_source", "")
        print(
            f"Scene {record['scene']} "
            f"[{record['start_timecode']} -> {record['end_timecode']}] ({source})"
        )
        print(record.get("transcript", "") or "(no speech)")
        image_path = record.get("image_path")
        if image_path:
            display(Image(filename=image_path, width=320))
        print()

In [ ]:
# Uncomment to compare with AdaptiveDetector
# adaptive_scenes = detect(str(video_path), AdaptiveDetector(), show_progress=True, start_in_scene=True)
# pd.DataFrame(scenes_to_records(adaptive_scenes))